# Rol 2. Ingeniero de transferencia (Tunchez)

**Task 1.2.** En este notebook se comparan cinco estrategias para adaptar un codificador preentrenado por un socio internacional. Con esos datos se recomienda una estrategia y se redacta la sección 2 del memo.

Se agrega la raíz del repositorio al `sys.path` para importar los módulos de `src/`. Los datos del enunciado están en `src/case_data.py`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import case_data as cd

## 1. Comparación de estrategias

Además de las columnas del enunciado, se calculan dos columnas. `ganancia_auc` es la mejora de AUC respecto a entrenar desde cero, y `auc_por_gpu_hora` es el AUC obtenido por cada hora de GPU. La tabla se ordena de mayor a menor AUC.

In [2]:
strategies = cd.transfer_strategies()
baseline = strategies.loc[strategies["estrategia"] == "Desde cero", "auc"].item()
strategies["ganancia_auc"] = strategies["auc"] - baseline
strategies["auc_por_gpu_hora"] = strategies["auc"] / strategies["gpu_horas"]
strategies.sort_values("auc", ascending=False).round(3)

,estrategia,auc,parametros_entrenables,gpu_horas,ganancia_auc,auc_por_gpu_hora
2,Ajuste fino de 2 bloques,0.87,0.180,1.5,0.09,0.58
4,LoRA rango 8,0.86,0.020,1.0,0.08,0.86
3,Ajuste fino completo,0.84,1.000,6.0,0.06,0.14
1,Extracción de características,0.82,0.005,0.5,0.04,1.64
0,Desde cero,0.78,1.000,6.0,0.00,0.13


### a. Estrategia elegida

Se elige el **ajuste fino de los dos últimos bloques con tasa de aprendizaje 10 veces menor**.

- **AUC.** Es la estrategia con mayor AUC (0.87), con una ganancia de 0.09 sobre entrenar desde cero.
- **GPU-horas.** Requiere 1.5 GPU-horas, frente a 6 del ajuste fino completo y del entrenamiento desde cero.
- **Parámetros entrenables.** Entrena solo el 18% de los parámetros y conserva fijos los bloques iniciales, que contienen patrones generales aprendidos con 3 millones de secuencias.

LoRA de rango 8 queda como alternativa cercana. Obtiene 0.86 con solo 2% de parámetros y 1 GPU-hora. La extracción de características es la más eficiente por GPU-hora (1.64), pero su AUC (0.82) es el más bajo entre las estrategias de transferencia.

**¿Por qué el ajuste fino completo rinde menos?** El ajuste fino completo actualiza el 100% de los parámetros con solo 1,200 positivos. Su pérdida de entrenamiento llega mucho más abajo que en el ajuste parcial, pero su AUC es menor (0.84). Esto indica sobreajuste. El modelo memoriza los pocos positivos y, al modificar todos los bloques, pierde parte de lo aprendido en el preentrenamiento (olvido catastrófico).

### b. Riesgos de reutilizar un modelo de otro corredor

| Riesgo | Cómo se verifica |
|---|---|
| **Escala de montos distinta.** El socio normalizó los montos con otra escala, por lo que el codificador recibiría valores en un rango que no aprendió. | Aplicar a los montos de RemesaSegura la misma normalización del socio y comparar los histogramas de entrada y de las representaciones del codificador entre ambos corredores. |
| **Cambio de distribución.** Los montos y patrones del corredor de origen no son los de Guatemala y Estados Unidos. | Evaluar el modelo con partición temporal sobre datos propios (ver Rol 3) y medir el recall por tipología de lavado del corredor. |
| **Sesgo hacia tipologías del otro corredor.** El codificador puede representar mejor patrones que no son frecuentes en este corredor. | Revisar con el equipo de cumplimiento los casos no detectados en validación, agrupados por tipología. |

### c. Sección 2 del memo

> **2. Transferencia.** Se recomienda el ajuste fino de los dos últimos bloques con tasa de aprendizaje 10 veces menor. Obtiene el mayor AUC (0.87) con 18% de parámetros entrenables y 1.5 GPU-horas, frente a 6 GPU-horas del ajuste completo, que sobreajusta los 1,200 positivos y baja a 0.84. Antes de usarlo debe homologarse la normalización de montos con la del socio. El AUC de 0.87 se obtuvo con partición aleatoria y debe confirmarse con partición temporal (Rol 3).